# Catalog Helpers Notebook

This notebook provides reusable utilities for schema and table management in the CDC pipeline.

## Functions
- `ensure_schema_exists(catalog, schema)` - Create schema if not exists
- `get_existing_schema(spark, table_fqn)` - Get existing Delta table schema
- `create_silver_table_orders(spark, table_fqn)` - Create orders Silver table
- `create_silver_table_products(spark, table_fqn)` - Create products Silver table
- `build_dynamic_merge_set(existing_columns, field_mapping)` - Build MERGE update/insert clauses

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, coalesce, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, DecimalType
import uuid

# Databricks utilities
def ensure_schema_exists(catalog: str, schema: str) -> None:
    """
    Ensure a schema exists in the specified catalog.
    
    Args:
        catalog: Unity Catalog name
        schema: Schema name to create
    """
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    print(f"Schema {catalog}.{schema} is ready")


def get_existing_schema(spark, table_fqn: str) -> StructType | None:
    """
    Get the schema of an existing Delta table.
    
    Args:
        spark: SparkSession
        table_fqn: Fully qualified table name (catalog.schema.table)
    
    Returns:
        StructType if table exists, None otherwise
    """
    try:
        df = spark.table(table_fqn)
        return df.schema
    except Exception as e:
        return None


def get_existing_columns(spark, table_fqn: str) -> set:
    """
    Get the set of column names from an existing Delta table.
    
    Args:
        spark: SparkSession
        table_fqn: Fully qualified table name (catalog.schema.table)
    
    Returns:
        Set of column names, or empty set if table doesn't exist
    """
    try:
        df = spark.table(table_fqn)
        return set(df.columns)
    except Exception:
        return set()


def create_silver_table_orders(spark, table_fqn: str) -> None:
    """
    Create the Silver orders table with schema for CDC upserts.
    
    Args:
        spark: SparkSession
        table_fqn: Fully qualified table name (catalog.schema.table)
    """
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            id INT,
            product_id INT,
            product_legacy STRING,
            price DECIMAL(12,2),
            created_at TIMESTAMP,
            last_inserted_dt TIMESTAMP,
            last_updated_dt TIMESTAMP
        ) USING DELTA
        """
    )
    print(f"Silver orders table {table_fqn} is ready")


def reconcile_silver_orders_table(spark, table_fqn: str, products_table_fqn: str) -> None:
    """
    Rewrite legacy Silver orders tables into the normalized schema used by the
    current CDC pipeline.
    """
    existing_schema = get_existing_schema(spark, table_fqn)
    if existing_schema is None:
        return

    existing_columns = {field.name: str(field.dataType) for field in existing_schema.fields}
    legacy_column = None
    for candidate in ["product_legacy", "product", "product_name"]:
        if candidate in existing_columns:
            legacy_column = candidate
            break

    needs_rewrite = (
        "product_id" not in existing_columns
        or existing_columns.get("price") != "DecimalType(12,2)"
        or "product_legacy" not in existing_columns
        or legacy_column in {"product", "product_name"}
    )
    if not needs_rewrite:
        return

    orders_df = spark.table(table_fqn).alias("o")
    if spark.catalog.tableExists(products_table_fqn):
        products_df = (
            spark.table(products_table_fqn)
            .select(
                col("id").alias("_product_id"),
                col("product_name").alias("_product_name")
            )
            .alias("p")
        )
        if legacy_column:
            joined_df = orders_df.join(
                products_df,
                col(f"o.{legacy_column}") == col("p._product_name"),
                "left"
            )
        else:
            joined_df = (
                orders_df
                .withColumn("_product_id", lit(None).cast("int"))
                .withColumn("_product_name", lit(None).cast("string"))
            )
    else:
        joined_df = (
            orders_df
            .withColumn("_product_id", lit(None).cast("int"))
            .withColumn("_product_name", lit(None).cast("string"))
        )

    source_product_id = col("o.product_id").cast("int") if "product_id" in existing_columns else lit(None).cast("int")
    source_product_legacy = col(f"o.{legacy_column}").cast("string") if legacy_column else lit(None).cast("string")

    migrated_df = joined_df.select(
        col("o.id").cast("int").alias("id"),
        coalesce(source_product_id, col("_product_id")).alias("product_id"),
        source_product_legacy.alias("product_legacy"),
        col("o.price").cast("decimal(12,2)").alias("price"),
        col("o.created_at").cast("timestamp").alias("created_at"),
        col("o.last_inserted_dt").cast("timestamp").alias("last_inserted_dt"),
        col("o.last_updated_dt").cast("timestamp").alias("last_updated_dt")
    )

    temp_table_fqn = f"{table_fqn}__migration_tmp_{uuid.uuid4().hex[:8]}"
    (
        migrated_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(temp_table_fqn)
    )
    spark.sql(f"CREATE OR REPLACE TABLE {table_fqn} AS SELECT * FROM {temp_table_fqn}")
    spark.sql(f"DROP TABLE IF EXISTS {temp_table_fqn}")
    print(f"Silver orders table {table_fqn} reconciled to normalized schema")


def create_silver_table_products(spark, table_fqn: str) -> None:
    """
    Create the Silver products table with schema for CDC upserts.
    
    Args:
        spark: SparkSession
        table_fqn: Fully qualified table name (catalog.schema.table)
    """
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {table_fqn} (
            id INT,
            product_name STRING,
            weight DECIMAL(10,2),
            color STRING,
            created_at TIMESTAMP,
            updated_at TIMESTAMP,
            last_inserted_dt TIMESTAMP,
            last_updated_dt TIMESTAMP
        ) USING DELTA
        """
    )
    print(f"Silver products table {table_fqn} is ready")


def build_merge_clauses(
    existing_columns: set,
    core_fields: list,
    include_inserted_dt: bool = True,
    include_updated_dt: bool = True
) -> tuple[dict, dict]:
    """
    Build dynamic MERGE update and insert clauses based on existing schema.
    
    Args:
        existing_columns: Set of existing column names in the target table
        core_fields: List of core field names to include in merge
        include_inserted_dt: Whether to include last_inserted_dt field
        include_updated_dt: Whether to include last_updated_dt field
    
    Returns:
        Tuple of (update_set dict, insert_values dict)
    """
    update_set = {}
    insert_values = {}
    
    # Core fields
    for field in core_fields:
        if field in existing_columns and field != "id":
            update_set[field] = f"s.{field}"
        if field in existing_columns:
            insert_values[field] = f"s.{field}"
    
    # Handle timestamp fields
    if include_inserted_dt and "last_inserted_dt" in existing_columns:
        update_set["last_inserted_dt"] = "COALESCE(s.event_time, t.last_inserted_dt)"
        insert_values["last_inserted_dt"] = "s.event_time"
    
    if include_updated_dt and "last_updated_dt" in existing_columns:
        update_set["last_updated_dt"] = "s.event_time"
        insert_values["last_updated_dt"] = "s.event_time"
    
    return update_set, insert_values


def execute_merge(
    spark,
    delta_table: DeltaTable,
    latest_df,
    update_set: dict,
    insert_values: dict,
    join_condition: str = "t.id = s.id",
    delete_condition: str = "s.op = 'd'",
    update_condition: str = "s.op <> 'd'"
) -> None:
    """
    Execute a MERGE operation with dynamic clauses.
    
    Args:
        spark: SparkSession
        delta_table: DeltaTable object for target
        latest_df: DataFrame with latest records to merge
        update_set: Dict of column -> value expressions for UPDATE
        insert_values: Dict of column -> value expressions for INSERT
        join_condition: Join condition string
        delete_condition: Condition for DELETE on matched
        update_condition: Condition for UPDATE on matched
    """
    merge_builder = (
        delta_table.alias("t")
        .merge(latest_df.alias("s"), join_condition)
    )
    
    # Build update clause
    if update_set:
        merge_builder = merge_builder.whenMatchedUpdate(
            condition=update_condition,
            set=update_set
        )
    
    # Build delete clause
    merge_builder = merge_builder.whenMatchedDelete(condition=delete_condition)
    
    # Build insert clause
    if insert_values:
        merge_builder = merge_builder.whenNotMatchedInsert(
            condition=update_condition,
            values=insert_values
        )
    
    merge_builder.execute()


## Usage Example

```python
# Run this helper notebook first to load functions
%run ./helpers/NB_catalog_helpers

# Use the helper functions
ensure_schema_exists(CONFIG['catalog'], CONFIG['silver_schema'])
create_silver_table_orders(spark, silver_table_fqn)

# For schema evolution in upsert
existing_columns = get_existing_columns(spark, silver_table_fqn)
update_set, insert_values = build_merge_clauses(existing_columns, core_fields)
execute_merge(spark, delta_table, latest, update_set, insert_values)
```